# Swin-S Transformer với APB (Adaptive Progressive Binarization)
## Assignment: So sánh Baseline vs APB trên ImageNette Dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.models as models
from torchvision import transforms, datasets
import math
import numpy as np
import time
from pathlib import Path
import requests
import tarfile

try:
    import thop
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'thop', '-q'], check=False)
    try:
        import thop
    except ImportError:
        pass

# QUICK_RUN = True  → chạy nhanh (~15 phút, CPU)
# QUICK_RUN = False → full run (GPU)
QUICK_RUN        = True
QUICK_SAMPLES    = 200
QUICK_VAL        = 100
QUICK_EPOCHS     = 3
QUICK_BATCH_SIZE = 4

print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
if QUICK_RUN:
    print(f"QUICK_RUN: {QUICK_SAMPLES} train / {QUICK_VAL} val | {QUICK_EPOCHS} epochs | BS={QUICK_BATCH_SIZE}")


Imports successful!
PyTorch version: 2.9.1+cpu
CUDA available:  False

QUICK_RUN mode ON — demo tren CPU (~15 min)
  Samples: 200 train / 100 val | Epochs: 3 | BS: 4


In [2]:
# ============================================================================
# CELL 2: APBLayer Class (Compatible với Swin-S)
# ============================================================================
class APBLayer(nn.Module):
    def __init__(self, layer_to_wrap: nn.Module):
        super().__init__()
        if not isinstance(layer_to_wrap, (nn.Linear, nn.Conv2d)):
            raise ValueError("APBLayer chỉ hỗ trợ nn.Linear và nn.Conv2d.")

        self.wrapped_layer = layer_to_wrap
        self.latent_weight = nn.Parameter(layer_to_wrap.weight.data.clone())
        # Keep bias as a regular attribute (not re-wrapped)
        self.bias = layer_to_wrap.bias
        # Remove the original weight so state_dict is clean
        if hasattr(self.wrapped_layer, 'weight'):
            del self.wrapped_layer.weight

        with torch.no_grad():
            weights = self.latent_weight.data
            initial_alpha = weights.abs().mean()
            initial_delta = 3.0 * weights.std().clamp(min=1e-5)
        self.alpha = nn.Parameter(torch.tensor(initial_alpha.item(), device=weights.device))
        self.delta = nn.Parameter(torch.tensor(initial_delta.item(), device=weights.device))

    # ── Expose .weight so that torchvision internals (e.g. ShiftedWindowAttention)
    #   that do `self.qkv.weight` still get the effective APB weight tensor.
    @property
    def weight(self):
        return self.get_effective_weight()

    def _compute_effective_weight(self):
        """STE: forward uses binarized weight, backward flows through latent_weight."""
        delta_clamped = self.delta.clamp(min=1e-8)
        w_hat = (self.latent_weight.abs() - self.alpha.abs()) / delta_clamped
        binarization_mask = (w_hat <= 1.0)
        sign_tensor = torch.sign(self.latent_weight)
        binarized_part = binarization_mask * sign_tensor * self.alpha.abs()
        full_precision_part = ~binarization_mask * self.latent_weight
        effective = binarized_part + full_precision_part
        # Straight-Through Estimator: gradients flow through latent_weight
        return self.latent_weight + (effective - self.latent_weight).detach()

    def forward(self, x):
        effective_weight = self._compute_effective_weight()
        if isinstance(self.wrapped_layer, nn.Linear):
            return F.linear(x, effective_weight, self.bias)
        elif isinstance(self.wrapped_layer, nn.Conv2d):
            return F.conv2d(
                x, effective_weight, self.bias,
                self.wrapped_layer.stride, self.wrapped_layer.padding,
                self.wrapped_layer.dilation, self.wrapped_layer.groups
            )

    def get_stats(self):
        with torch.no_grad():
            total_weights = self.latent_weight.numel()
            threshold = self.alpha.abs() + self.delta.clamp(min=1e-8)
            num_binary = (self.latent_weight.abs() <= threshold).sum().item()
            return {
                "alpha": self.alpha.item(),
                "delta": self.delta.item(),
                "percent_binary": (num_binary / total_weights) * 100,
            }

    def get_effective_weight(self):
        """Detached effective weight for stats / saving."""
        with torch.no_grad():
            delta_clamped = self.delta.clamp(min=1e-8)
            threshold = self.alpha.abs() + delta_clamped
            binarization_mask = self.latent_weight.abs() <= threshold
            sign_tensor = torch.ones_like(self.latent_weight)
            sign_tensor[self.latent_weight < 0] = -1
            binarized_part     = torch.where(binarization_mask, sign_tensor * self.alpha, torch.zeros_like(self.latent_weight))
            full_precision_part = torch.where(~binarization_mask, self.latent_weight, torch.zeros_like(self.latent_weight))
            return binarized_part + full_precision_part

print("APBLayer class defined!")


APBLayer class defined!


In [ ]:
# Cell 2b: FIMAQLinear — Post-Training Quantization (PTQ)
# Nguồn: ShiheWang/FIMA-Q (CVPR 2025) — github.com/ShiheWang/FIMA-Q
# Quantize weight (symmetric, per output-channel) + activation (asymmetric, per tensor)
# Mixed-precision: attn layers W4A4, MLP layers W8A8
# Mode: 'raw' | 'calibration' | 'quant_forward'

class FIMAQLinear(nn.Linear):
    def __init__(self, in_features: int, out_features: int,
                 bias: bool = True, w_bits: int = 8, a_bits: int = 8):
        super().__init__(in_features, out_features, bias)
        self.w_bits    = w_bits
        self.a_bits    = a_bits
        self.mode      = 'raw'          # 'raw' | 'calibration' | 'quant_forward'
        self.calibrated = False

        self.raw_input  = None
        self.raw_output = None

        self.register_buffer('w_scale',       torch.ones(out_features, 1))
        self.register_buffer('a_scale',       torch.tensor(1.0))
        self.register_buffer('a_zero_point',  torch.tensor(0.0))

    def _quant_weight(self) -> torch.Tensor:
        n_levels = 2 ** (self.w_bits - 1) - 1
        scale = self.w_scale.to(self.weight.device)
        w_q   = torch.clamp(torch.round(self.weight / scale), -n_levels, n_levels)
        return w_q * scale

    def _quant_act(self, x: torch.Tensor) -> torch.Tensor:
        n_levels   = 2 ** self.a_bits - 1
        scale      = self.a_scale.to(x.device)
        zero_point = self.a_zero_point.to(x.device)
        x_q = torch.clamp(torch.round(x / scale + zero_point), 0, n_levels)
        return (x_q - zero_point) * scale

    def calibrate(self):
        if self.raw_input is None:
            return
        data    = torch.cat(self.raw_input, dim=0).float()
        x_min   = data.min()
        x_max   = data.max()
        n_lev   = 2 ** self.a_bits - 1
        a_scale = (x_max - x_min) / n_lev
        a_zp    = torch.round(-x_min / a_scale)
        self.a_scale.fill_(a_scale.item() if a_scale.item() > 1e-8 else 1e-8)
        self.a_zero_point.fill_(torch.clamp(a_zp, 0, n_lev).item())

        w       = self.weight.data.float()
        w_abs   = w.abs().max(dim=1, keepdim=True).values
        n_lev_w = float(2 ** (self.w_bits - 1) - 1)
        self.w_scale.data.copy_((w_abs / n_lev_w).clamp(min=1e-8))

        self.calibrated  = True
        self.raw_input   = None

    def compute_fisher_score(self) -> torch.Tensor:
        if self.weight.grad is None:
            return torch.zeros_like(self.weight.data)
        fisher = (self.weight.data.abs() * self.weight.grad.abs())
        fisher = (fisher - fisher.min()) / (fisher.max() - fisher.min() + 1e-8)
        return fisher.detach()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.mode == 'calibration':
            x_detach = x.detach().cpu()
            if self.raw_input is None:
                self.raw_input = [x_detach]
            else:
                self.raw_input.append(x_detach)
                if len(self.raw_input) > 8:
                    self.raw_input.pop(0)
            return F.linear(x, self.weight, self.bias)

        if self.mode == 'quant_forward':
            x_q = self._quant_act(x)
            w_q = self._quant_weight()
            return F.linear(x_q, w_q, self.bias)

        return F.linear(x, self.weight, self.bias)


def get_fimaq_compression_stats(model: nn.Module) -> dict:
    w_orig_bits = 0; w_comp_bits = 0
    a_orig_bits = 0; a_comp_bits = 0
    n_layers = n_4bit = n_8bit = 0

    for _, m in model.named_modules():
        if not isinstance(m, FIMAQLinear):
            continue
        n_w = m.weight.numel()
        w_orig_bits += n_w * 32
        w_comp_bits += n_w * m.w_bits
        a_orig_bits += m.in_features * 32
        a_comp_bits += m.in_features * m.a_bits
        n_layers += 1
        if m.w_bits <= 4: n_4bit += 1
        else:             n_8bit += 1

    if n_layers == 0:
        return {'n_fimaq_layers': 0, 'layers_4bit': 0, 'layers_8bit': 0,
                'cr_weight': 1.0, 'cr_activation': 1.0, 'compression_ratio': 1.0}

    cr_weight     = w_orig_bits / w_comp_bits if w_comp_bits > 0 else 1.0
    cr_activation = a_orig_bits / a_comp_bits if a_comp_bits > 0 else 1.0
    cr_combined   = (w_orig_bits + a_orig_bits) / (w_comp_bits + a_comp_bits)
    return {
        'n_fimaq_layers':   n_layers,
        'layers_4bit':      n_4bit,
        'layers_8bit':      n_8bit,
        'cr_weight':        round(cr_weight, 2),
        'cr_activation':    round(cr_activation, 2),
        'compression_ratio': round(cr_combined, 2),
    }


print("FIMAQLinear defined!")


FIMAQLinear (PTQ — FIMA-Q CVPR 2025) and helpers defined!
Source: https://github.com/ShiheWang/FIMA-Q


In [ ]:
# Cell 2c: APBFIMAQLayer — APB kết hợp Fisher Information từ FIMA-Q
#
# Thay magnitude threshold bằng Fisher score để chọn FP32 weights:
#   APB thuần:   binarize nếu |w_hat| <= 1
#   APB+FIMA-Q:  binarize nếu |w_hat| <= 1 VÀ Fisher(w) thấp
#                → top fisher_keep_ratio (10%) Fisher weights được giữ FP32
# Activation quantization được transfer từ FIMA-Q (nén cả weight + activation)

class APBFIMAQLayer(APBLayer):
    """
    APB binarization với Fisher-guided FP32 protection + FIMA-Q activation quantization.

    Args
    ----
    layer_to_wrap       : nn.Linear or nn.Conv2d
    fisher_score        : per-weight Fisher importance (same shape as weight), [0,1]
    fisher_keep_ratio   : fraction of weights (highest Fisher) protected as FP32
    a_scale / a_zp      : pre-calibrated activation quantizer params from FIMA-Q
    a_bits              : activation bit-width
    """

    def __init__(self, layer_to_wrap: nn.Module,
                 fisher_score:       torch.Tensor = None,
                 fisher_keep_ratio:  float = 0.10,
                 a_scale:            float = 1.0,
                 a_zp:               float = 0.0,
                 a_bits:             int   = 8):
        super().__init__(layer_to_wrap)

        self.a_bits            = a_bits
        self.fisher_keep_ratio = fisher_keep_ratio
        self.act_calibrated    = (a_scale != 1.0)

        if fisher_score is not None:
            thresh  = torch.quantile(fisher_score.flatten(),
                                     1.0 - fisher_keep_ratio)
            fp_mask = (fisher_score >= thresh)
        else:
            fp_mask = torch.zeros_like(self.latent_weight, dtype=torch.bool)
        self.register_buffer('fp_mask', fp_mask)

        self.register_buffer('a_scale_buf', torch.tensor(float(a_scale)))
        self.register_buffer('a_zp_buf',    torch.tensor(float(a_zp)))

    def _compute_effective_weight(self) -> torch.Tensor:
        delta_c = self.delta.clamp(min=1e-8)
        w_hat   = (self.latent_weight.abs() - self.alpha.abs()) / delta_c
        apb_can_binarize  = (w_hat <= 1.0)
        actually_binarize = apb_can_binarize & (~self.fp_mask)

        sign  = torch.sign(self.latent_weight)
        bin_  = actually_binarize  * sign * self.alpha.abs()
        fp_   = (~actually_binarize) * self.latent_weight
        eff   = bin_ + fp_
        return self.latent_weight + (eff - self.latent_weight).detach()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.act_calibrated:
            n = 2 ** self.a_bits - 1
            x = ((x / self.a_scale_buf + self.a_zp_buf)
                 .round().clamp(0, n) - self.a_zp_buf) * self.a_scale_buf
        return super().forward(x)

    def get_stats(self) -> dict:
        base = super().get_stats()
        with torch.no_grad():
            n      = self.latent_weight.numel()
            delta_c = self.delta.clamp(min=1e-8)
            w_hat   = (self.latent_weight.abs() - self.alpha.abs()) / delta_c
            n_eff_bin   = ((w_hat <= 1.0) & ~self.fp_mask).sum().item()
            n_fp_guard  = self.fp_mask.sum().item()
        base['fisher_fp_protected']     = n_fp_guard
        base['fisher_fp_protected_pct'] = round(n_fp_guard  / n * 100, 2)
        base['effective_binary_pct']    = round(n_eff_bin   / n * 100, 2)
        base['act_quantized']           = self.act_calibrated
        base['a_bits']                  = self.a_bits
        return base


def get_apb_fimaq_compression_stats(model) -> dict:
    total = bin_w = fp_w = act_bits_total = 0
    for _, m in model.named_modules():
        if isinstance(m, APBFIMAQLayer):
            st = m.get_stats()
            n  = m.latent_weight.numel()
            n_bin = int(st['effective_binary_pct'] / 100 * n)
            n_fp  = n - n_bin
            total         += n
            bin_w         += n_bin
            fp_w          += n_fp
            act_bits_total += n * m.a_bits
    if total == 0:
        return {}
    w_bits_compressed = bin_w * 1 + fp_w * 32
    orig_w_bits       = total * 32
    cr_weight         = orig_w_bits / w_bits_compressed
    cr_act            = (total * 32) / act_bits_total if act_bits_total > 0 else 1.0
    return {
        'total_weights':      total,
        'binary_weights':     bin_w,
        'fp_weights':         fp_w,
        'binary_pct':         round(bin_w / total * 100, 2),
        'cr_weight':          round(cr_weight, 2),
        'cr_activation':      round(cr_act, 2),
        'size_reduction_pct': round((1 - 1 / cr_weight) * 100, 2),
    }

print("APBFIMAQLayer defined!")


APBFIMAQLayer + get_apb_fimaq_compression_stats defined!


In [5]:
# ============================================================================
# CELL 3: Apply APB Function (Optimized cho Swin Transformer)
# ============================================================================
def apply_apb(model: nn.Module, skip_first_conv=True, skip_last_linear=True):
    """
    Apply APB to Swin Transformer layers
    - Skip patch embedding conv
    - Skip final classification head
    - Apply to all Linear layers in attention và MLP blocks
    """
    conv_layers = [(name, module) for name, module in model.named_modules() if isinstance(module, nn.Conv2d)]
    first_conv_name = conv_layers[0][0] if conv_layers and skip_first_conv else None
    
    linear_layers = [(name, module) for name, module in model.named_modules() if isinstance(module, nn.Linear)]
    last_linear_name = linear_layers[-1][0] if linear_layers and skip_last_linear else None

    applied_count = 0
    for name, module in list(model.named_modules()):
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            is_first_conv = (name == first_conv_name) and isinstance(module, nn.Conv2d)
            is_last_linear = (name == last_linear_name) and isinstance(module, nn.Linear)
            
            # Skip patch embedding và classifier head
            if is_first_conv or is_last_linear:
                print(f"⊗ Skipping: {name}")
                continue
                
            print(f"✓ Applying APB to: {name}")
            parent_name = '.'.join(name.split('.')[:-1])
            child_name = name.split('.')[-1]
            parent_module = model
            if parent_name:
                for part in parent_name.split('.'):
                    parent_module = getattr(parent_module, part)
            setattr(parent_module, child_name, APBLayer(module))
            applied_count += 1

    print(f"\n→ Applied APB to {applied_count} layers")
    return model

print("apply_apb function defined!")

apply_apb function defined!


In [ ]:
# Cell 3b: apply_fimaq / calibrate_fimaq / compute_fisher_for_model / apply_apb_fimaq

import copy

def apply_fimaq(model: nn.Module,
                w_bits: int = 8, a_bits: int = 8,
                skip_last_linear: bool = True) -> nn.Module:
    """
    Thay thế nn.Linear (trừ head) bằng FIMAQLinear.
    Mixed-precision: attn (qkv/proj) → W4A4, MLP (fc1/fc2) → W8A8.
    Set mode='calibration', gọi calibrate_fimaq() sau khi chạy calibration data.
    """
    linears   = [(n, m) for n, m in model.named_modules() if isinstance(m, nn.Linear)]
    head_name = linears[-1][0] if linears and skip_last_linear else None

    def _bits(name: str):
        ll = name.lower()
        if any(k in ll for k in ('qkv', 'proj', 'attn')):
            return 4, 4
        return w_bits, a_bits

    count = 0
    for name, module in list(model.named_modules()):
        if not isinstance(module, nn.Linear):
            continue
        if name == head_name:
            print(f"  ⊗ Skip (head): {name}")
            continue

        wb, ab = _bits(name)
        new = FIMAQLinear(module.in_features, module.out_features,
                          bias=module.bias is not None, w_bits=wb, a_bits=ab)
        new.weight.data.copy_(module.weight.data)
        if module.bias is not None:
            new.bias.data.copy_(module.bias.data)
        new.mode = 'calibration'

        parent = model
        parts  = name.split('.')
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], new.to(module.weight.device))
        count += 1
        print(f"  ✓ FIMAQLinear W{wb}A{ab}: {name}")

    print(f"\n  → Replaced {count} nn.Linear with FIMAQLinear (mode=calibration)")
    return model


def calibrate_fimaq(model: nn.Module, calib_loader,
                    device, n_batches: int = 4) -> nn.Module:
    """
    Calibrate FIMA-Q: thu thập stats min/max từ n_batches,
    tính scale/zero_point, chuyển sang mode='quant_forward'.
    """
    model.eval()
    with torch.no_grad():
        for i, (imgs, _) in enumerate(calib_loader):
            if i >= n_batches:
                break
            model(imgs.to(device))
    print(f"  Collected calibration data from {min(n_batches, i+1)} batch(es)")

    n_cal = 0
    for name, m in model.named_modules():
        if isinstance(m, FIMAQLinear) and not m.calibrated:
            if m.raw_input is not None:
                m.calibrate()
                n_cal += 1
    print(f"  Calibrated {n_cal} FIMAQLinear layer(s)")

    for _, m in model.named_modules():
        if isinstance(m, FIMAQLinear) and m.calibrated:
            m.mode = 'quant_forward'

    return model


def compute_fisher_for_model(model: nn.Module, calib_loader,
                             device, criterion,
                             n_batches: int = 4) -> dict:
    """
    Tính Fisher Information score cho các FIMAQLinear layer.
    F_i ≈ |w_i · ∂L/∂w_i| (first-order approximation).
    """
    for _, m in model.named_modules():
        if isinstance(m, FIMAQLinear):
            m.mode = 'raw'

    model.train()
    model.zero_grad()

    for i, (imgs, labels) in enumerate(calib_loader):
        if i >= 1:
            break
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(model(imgs), labels)
        loss.backward()

    fisher_scores = {}
    for name, m in model.named_modules():
        if isinstance(m, FIMAQLinear):
            fisher_scores[name] = m.compute_fisher_score().clone().detach()

    model.zero_grad()
    model.eval()

    for _, m in model.named_modules():
        if isinstance(m, FIMAQLinear) and m.calibrated:
            m.mode = 'quant_forward'

    print(f"  Computed Fisher scores for {len(fisher_scores)} layer(s)")
    return fisher_scores


def apply_apb_fimaq(model_fimaq: nn.Module,
                    fisher_scores: dict,
                    fisher_keep_ratio: float = 0.10,
                    a_bits: int = 8) -> nn.Module:
    """
    Xây APB+FIMA-Q từ model FIMA-Q đã calibrate.
    Mỗi FIMAQLinear → APBFIMAQLayer:
      - Fisher mask: top fisher_keep_ratio weights được giữ FP32
      - Activation quantization từ FIMA-Q calibration được giữ nguyên
    """
    count = 0
    for name, m in list(model_fimaq.named_modules()):
        if not isinstance(m, FIMAQLinear):
            continue

        fisher = fisher_scores.get(name, None)

        tmp = nn.Linear(m.in_features, m.out_features, bias=m.bias is not None)
        tmp.weight.data.copy_(m.weight.data)
        if m.bias is not None:
            tmp.bias.data.copy_(m.bias.data)

        a_sc = m.a_scale.item()      if m.calibrated else 1.0
        a_zp = m.a_zero_point.item() if m.calibrated else 0.0

        new_layer = APBFIMAQLayer(
            tmp,
            fisher_score      = fisher.to(tmp.weight.device) if fisher is not None else None,
            fisher_keep_ratio = fisher_keep_ratio,
            a_scale           = a_sc,
            a_zp              = a_zp,
            a_bits            = a_bits,
        )

        parent = model_fimaq
        parts  = name.split('.')
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], new_layer)
        count += 1
        print(f"  ✓ APBFIMAQLayer: {name}  "
              f"(fp_protected={new_layer.fp_mask.sum().item():,})")

    print(f"\n  → Created {count} APBFIMAQLayer(s) "
          f"(fisher_keep_ratio={fisher_keep_ratio:.0%}, a_bits={a_bits})")
    return model_fimaq


print("apply_fimaq / calibrate_fimaq / compute_fisher_for_model / apply_apb_fimaq defined!")


apply_fimaq / calibrate_fimaq / compute_fisher_for_model / apply_apb_fimaq defined!
Layer analysis documented in cell comments above.


In [7]:
# ============================================================================
# CELL 4: Model Size, Parameters, FLOPs, and Compression Metrics
# ============================================================================

# ── FLOPs measurement via thop ────────────────────────────────────────────
def get_flops(model, input_size=(1, 3, 224, 224), device='cpu'):
    """
    Measure FLOPs (multiply-accumulate operations × 2) using the thop library.
    Returns (flops_int, flops_str) where flops_str is human-readable (e.g. '8.74G').
    If thop is not installed, returns (None, 'N/A (pip install thop)').
    """
    try:
        from thop import profile, clever_format
        model.eval()
        dummy = torch.randn(*input_size).to(device)
        with torch.no_grad():
            macs, _ = profile(model, inputs=(dummy,), verbose=False)
        flops = macs * 2
        flops_str = clever_format([flops], "%.2f")[0]
        return int(flops), flops_str
    except ImportError:
        return None, "N/A — run: pip install thop"
    except Exception as e:
        return None, f"N/A ({e})"


# ── Parameter counting ────────────────────────────────────────────────────
def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ── Model size (MB) ───────────────────────────────────────────────────────
def get_model_size(model):
    """Estimate in-memory size in MB (parameters + buffers)."""
    p_bytes = sum(p.nelement() * p.element_size() for p in model.parameters())
    b_bytes = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (p_bytes + b_bytes) / 1024 / 1024


# ── APB-only compression stats ────────────────────────────────────────────
def get_apb_compression_stats(model):
    """
    Compression ratio for APB-only model (weight binarization, no activation quantization).
    Consistent with rebuildapb.ipynb source.
    """
    total = binary = fp = 0
    for _, m in model.named_modules():
        if isinstance(m, APBLayer) and not isinstance(m, APBFIMAQLayer):
            n = m.latent_weight.numel()
            n_bin = int(m.get_stats()['percent_binary'] / 100 * n)
            total  += n
            binary += n_bin
            fp     += (n - n_bin)
    if total == 0:
        return {'total_weights': 0, 'binary_weights': 0, 'fp_weights': 0,
                'binary_percentage': 0.0, 'compression_ratio': 1.0, 'size_reduction': 0.0}
    orig_bits  = total  * 32
    comp_bits  = binary * 1 + fp * 32
    cr         = orig_bits / comp_bits if comp_bits > 0 else 1.0
    return {
        'total_weights':    total,
        'binary_weights':   binary,
        'fp_weights':       fp,
        'binary_percentage': round(binary / total * 100, 2),
        'compression_ratio': round(cr, 2),
        'size_reduction':    round((1 - 1 / cr) * 100, 2),
    }


print("Model metrics functions defined!")
print("  count_parameters  |  get_model_size  |  get_flops  |  get_apb_compression_stats")


Model metrics functions defined!
  count_parameters  |  get_model_size  |  get_flops  |  get_apb_compression_stats


In [8]:
# ============================================================================
# CELL 5: Evaluation với Speed Metrics
# ============================================================================
def evaluate(model, data_loader, criterion, device, measure_speed=False):
    """Evaluate model with optional speed measurement"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    if measure_speed:
        inference_times = []
        
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            if measure_speed:
                torch.cuda.synchronize() if torch.cuda.is_available() else None
                start = time.time()
                outputs = model(inputs)
                torch.cuda.synchronize() if torch.cuda.is_available() else None
                inference_times.append(time.time() - start)
            else:
                outputs = model(inputs)
            
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(data_loader)
    accuracy = 100 * correct / total
    
    result = {"loss": avg_loss, "accuracy": accuracy}
    
    if measure_speed:
        avg_time = np.mean(inference_times)
        std_time = np.std(inference_times)
        throughput = len(data_loader.dataset) / sum(inference_times)
        result.update({
            "avg_inference_time": avg_time,
            "std_inference_time": std_time,
            "throughput": throughput
        })
    
    return result

print("Evaluate function defined!")

Evaluate function defined!


In [ ]:
# Cell 6: Data Preparation — ImageNette (fastai/imagenette, 10-class ImageNet subset)
# Pretrained: torchvision swin_s(weights='DEFAULT') = Swin_S_Weights.IMAGENET1K_V1 (83.1% top-1)
# APB: https://www.kaggle.com/code/dyhngg/rebuildapb
# FIMA-Q: https://github.com/ShiheWang/FIMA-Q (Wu et al., CVPR 2025)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

import pathlib as _pl
_candidate1 = _pl.Path('./data/imagenette2-320')
_candidate2 = _pl.Path('../data/imagenette2-320')

if (_candidate1 / 'train').exists() and any((_candidate1 / 'train').iterdir()):
    data_dir = _pl.Path('./data')
elif (_candidate2 / 'train').exists() and any((_candidate2 / 'train').iterdir()):
    data_dir = _pl.Path('../data')
    print(f"  Data: {data_dir.resolve()}")
else:
    data_dir = _pl.Path('./data')

imagenette_dir = data_dir / 'imagenette2-320'

if not imagenette_dir.exists():
    print("Downloading ImageNette...")
    url = 'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz'
    tar_path = data_dir / 'imagenette2-320.tgz'
    data_dir.mkdir(exist_ok=True)
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    with open(tar_path, 'wb') as f:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total_size > 0:
                print(f"\r{downloaded/1024/1024:.1f}/{total_size/1024/1024:.1f} MB", end='')
    print("\nExtracting...")
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(data_dir)
    tar_path.unlink()
    print("Done!")
else:
    print(f"ImageNette found: {imagenette_dir}")

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset_full = datasets.ImageFolder(imagenette_dir / 'train', transform=train_transform)
val_dataset_full   = datasets.ImageFolder(imagenette_dir / 'val',   transform=val_transform)

if QUICK_RUN:
    from torch.utils.data import Subset
    import random
    random.seed(42)
    train_indices = random.sample(range(len(train_dataset_full)), min(QUICK_SAMPLES, len(train_dataset_full)))
    val_indices   = random.sample(range(len(val_dataset_full)),   min(QUICK_VAL,     len(val_dataset_full)))
    train_dataset = Subset(train_dataset_full, train_indices)
    val_dataset   = Subset(val_dataset_full,   val_indices)
    batch_size    = QUICK_BATCH_SIZE
    num_workers   = 0
else:
    train_dataset = train_dataset_full
    val_dataset   = val_dataset_full
    batch_size    = 32
    num_workers   = 2

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=num_workers, pin_memory=False
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=False
)

print(f"\nTrain: {len(train_dataset)} samples | Val: {len(val_dataset)} samples | BS: {batch_size}")
print(f"Classes: {train_dataset_full.classes}")


DATA & MODEL SOURCES
  Dataset  : ImageNette (10-class subset of ImageNet)
  URL      : https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
  Credit   : fastai / Jeremy Howard (2019) — github.com/fastai/imagenette

  Pretrained weights: torchvision Swin_S_Weights.IMAGENET1K_V1
  Top-1 accuracy    : 83.1%  (ImageNet-1K)
  Ref: https://pytorch.org/vision/stable/models/swin_transformer.html

  APB source : https://www.kaggle.com/code/dyhngg/rebuildapb
  FIMA-Q src : https://github.com/ShiheWang/FIMA-Q  (CVPR 2025)
Using device: cpu
  Using data path: C:\Users\user\OneDrive\Desktop\AI\Swin-S_github\data
 ImageNette dataset found at ..\data\imagenette2-320

 Dataset ready!
  Training samples:   200
  Validation samples: 100
  Batch size:         4
  Batches/epoch:      train=50, val=25
  Classes:            ['n01440764', 'n02102040', 'n02979186', 'n03000684', 'n03028079', 'n03394916', 'n03417042', 'n03425413', 'n03445777', 'n03888257']


In [ ]:
# Cell 7: Load Swin-S và khởi tạo các models
import copy

num_classes = len(train_dataset_full.classes)
print(f"Classes: {num_classes}")

# Model gốc: backbone ImageNet-1K weights, head 10-class random (không train) — dùng làm tham chiếu
print("\n[Model gốc]")
model_pretrained = models.swin_s(weights='DEFAULT')
model_pretrained.head = nn.Linear(model_pretrained.head.in_features, num_classes)
pt_total_params, _ = count_parameters(model_pretrained)
print(f"  {pt_total_params:,} params  |  {get_model_size(model_pretrained):.1f} MB")

# Baseline Swin-S — finetune trên ImageNette
model_baseline = models.swin_s(weights='DEFAULT')
model_baseline.head = nn.Linear(model_baseline.head.in_features, num_classes)
total_params, trainable_params = count_parameters(model_baseline)
model_size = get_model_size(model_baseline)
print(f"\n[Baseline] {total_params:,} params  |  {model_size:.1f} MB")

# APB Swin-S
# Skip: Conv2d đầu (patch embed) + Linear cuối (head)
print("\n[APB] Applying APB...")
model_apb = models.swin_s(weights='DEFAULT')
model_apb.head = nn.Linear(model_apb.head.in_features, num_classes)
model_apb = apply_apb(model_apb, skip_first_conv=True, skip_last_linear=True)
apb_total_params, apb_trainable_params = count_parameters(model_apb)
apb_model_size = get_model_size(model_apb)
print(f"  {apb_total_params:,} params  |  {apb_model_size:.1f} MB")

# FIMA-Q Swin-S — PTQ, calibration chạy ở Cell 11
# Mixed-precision: attn W4A4, MLP W8A8 | Skip: head
print("\n[FIMA-Q] Applying FIMA-Q (mode=calibration)...")
model_fimaq = models.swin_s(weights='DEFAULT')
model_fimaq.head = nn.Linear(model_fimaq.head.in_features, num_classes)
model_fimaq = apply_fimaq(model_fimaq, w_bits=8, a_bits=8, skip_last_linear=True)
fimaq_total_params, _ = count_parameters(model_fimaq)
fimaq_model_size = get_model_size(model_fimaq)
fimaq_init_stats = get_fimaq_compression_stats(model_fimaq)
print(f"  {fimaq_total_params:,} params  |  {fimaq_model_size:.1f} MB")
print(f"  Layers: {fimaq_init_stats['n_fimaq_layers']} (4-bit: {fimaq_init_stats['layers_4bit']}, 8-bit: {fimaq_init_stats['layers_8bit']})")
print(f"  CR (W+A): {fimaq_init_stats['compression_ratio']:.2f}x  (activate sau calibration)")

# APB+FIMA-Q sẽ được tạo ở Cell 12 (sau khi FIMA-Q calibrate + Fisher)
print("\n[APB+FIMA-Q] → sẽ tạo ở Cell 12")

model_pretrained.to(device)
model_baseline.to(device)
model_apb.to(device)
model_fimaq.to(device)

print(f"\nAll models on {device}")


LOADING SWIN-S (torchvision, ImageNet-1K DEFAULT weights)

REFERENCE MODEL (model gốc)
  torchvision Swin-S + ImageNet-1K weights, no ImageNette adaptation
  Head: nn.Linear(in_features, 10) — randomly initialized, NOT trained
  Parameters: 48,844,948  |  Size: 186.8 MB

  Baseline Swin-S:
    Parameters: 48,844,948  |  Size: 186.8 MB

APPLYING APB — Adaptive Progressive Binarization
  Source: rebuildapb.ipynb (Kaggle: dyhngg)
  Skip: features.0.0 (Conv2d, patch embed) + head (Linear)
  Compression: WEIGHT ONLY (no activation quantization)
⊗ Skipping: features.0.0
✓ Applying APB to: features.1.0.attn.qkv
✓ Applying APB to: features.1.0.attn.proj
✓ Applying APB to: features.1.0.mlp.0
✓ Applying APB to: features.1.0.mlp.3
✓ Applying APB to: features.1.1.attn.qkv
✓ Applying APB to: features.1.1.attn.proj
✓ Applying APB to: features.1.1.mlp.0
✓ Applying APB to: features.1.1.mlp.3
✓ Applying APB to: features.2.reduction
✓ Applying APB to: features.3.0.attn.qkv
✓ Applying APB to: features.3.

In [13]:
# Cell 7b: Phân tích layer — xác định những layer có thể áp dụng APB (và layer nào không)
#
# Tiêu chí APB applicable:
#   - phải là nn.Linear hoặc nn.Conv2d
#   - không phải Conv2d đầu (patch embed) → skip_first_conv=True
#   - không phải Linear cuối (head)       → skip_last_linear=True
#
# Tiêu chí FIMA-Q applicable:
#   - phải là nn.Linear (không wrap Conv2d, LayerNorm)
#   - không phải classifier head

print("="*92)
print("LAYER ANALYSIS — APB & FIMA-Q APPLICABILITY ON SWIN-S")
print("="*92)

all_convs   = [(n, m) for n, m in model_baseline.named_modules() if isinstance(m, nn.Conv2d)]
all_linears = [(n, m) for n, m in model_baseline.named_modules() if isinstance(m, nn.Linear)]
first_conv  = all_convs[0][0]    if all_convs   else None
last_linear = all_linears[-1][0] if all_linears else None

LAYER_REASONS = {
    'FIRST_CONV': (
        "First Conv2d (patch embedding). Encodes raw pixels; binarizing "
        "destroys low-level feature extraction irreversibly."),
    'LAST_LINEAR': (
        "Last Linear (classifier head). Errors map directly to "
        "wrong class predictions; extremely sensitive to quantization."),
    'LAYERNORM': (
        "LayerNorm. Scale/bias critical for normalisation stability. "
        "Quantizing causes severe accuracy loss. Not supported by APBLayer/FIMAQLinear."),
    'CONV_FIMAQ': (
        "Conv2d (non-first). FIMAQLinear wraps nn.Linear only. "
        "APB can wrap Conv2d, but no Conv2d inside Swin-S transformer blocks."),
}

rows = []
for name, module in model_baseline.named_modules():
    t = type(module)
    if not isinstance(module, (nn.Linear, nn.Conv2d, nn.LayerNorm)):
        continue

    n_params = sum(p.numel() for p in module.parameters())
    apb_ok = fimaq_ok = True
    reason = "nn.Linear in transformer block — safe to binarize (APB) and quantize (FIMA-Q)."
    skip_tag = ""

    if isinstance(module, nn.Conv2d):
        if name == first_conv:
            apb_ok = fimaq_ok = False
            reason   = LAYER_REASONS['FIRST_CONV']
            skip_tag = "SKIP_ALL"
        else:
            fimaq_ok = False
            reason   = LAYER_REASONS['CONV_FIMAQ']
            skip_tag = "SKIP_FIMAQ"

    elif isinstance(module, nn.Linear):
        if name == last_linear:
            apb_ok = fimaq_ok = False
            reason   = LAYER_REASONS['LAST_LINEAR']
            skip_tag = "SKIP_ALL"

    elif isinstance(module, nn.LayerNorm):
        apb_ok = fimaq_ok = False
        reason   = LAYER_REASONS['LAYERNORM']
        skip_tag = "SKIP_ALL"

    rows.append(dict(name=name, type=t.__name__, apb=apb_ok,
                     fimaq=fimaq_ok, n_params=n_params,
                     reason=reason, skip=skip_tag))

# ── CAN apply APB ─────────────────────────────────────────────────────────
can_apb = [r for r in rows if r['apb']]
print(f"\n{'[CAN APPLY APB]':-^90}")
print(f"  {'Layer Name':<58} {'Type':<12} {'Params':>10}")
print("  " + "-"*83)
cats = {}
for r in can_apb:
    leaf = r['name'].rsplit('.', 1)[-1]
    cats[leaf] = cats.get(leaf, 0) + 1
    print(f"  {r['name']:<58} {r['type']:<12} {r['n_params']:>10,}")
total_apb_params = sum(r['n_params'] for r in can_apb)
print(f"\n  Subtotal: {len(can_apb)} layers  |  {total_apb_params:,} params")
print(f"  Layer-type breakdown: " + "  ".join(f"{k}×{v}" for k, v in sorted(cats.items())))

attn_layers = [r for r in can_apb if any(k in r['name'] for k in ('qkv', 'proj', 'attn'))]
mlp_layers  = [r for r in can_apb if any(k in r['name'] for k in ('fc1', 'fc2', 'mlp'))]
other       = [r for r in can_apb if r not in attn_layers and r not in mlp_layers]
print(f"  Attention proj: {len(attn_layers)}  |  MLP: {len(mlp_layers)}  |  Other: {len(other)}")

# ── CANNOT apply APB ──────────────────────────────────────────────────────
skip_apb = [r for r in rows if not r['apb']]
print(f"\n{'[CANNOT APPLY APB — SKIP]':-^90}")
print(f"  {'Layer Name':<40} {'Type':<14} {'Handling'}")
print("  " + "-"*88)
handling_map = {
    'SKIP_ALL':    'skip_first_conv=True / skip_last_linear=True in apply_apb() / apply_fimaq()',
    'SKIP_FIMAQ':  'APB wraps Conv2d; FIMAQLinear skips (consistent with repo)',
    '':            'Type not supported — not wrapped by APBLayer or FIMAQLinear',
}
for r in skip_apb:
    handling = handling_map.get(r['skip'], handling_map[''])
    print(f"  {r['name']:<40} {r['type']:<14} {handling}")
    print(f"  {'':40} {'':14} Reason: {r['reason'][:72]}")

# ── FIMA-Q applicability ──────────────────────────────────────────────────
can_fimaq = [r for r in rows if r['fimaq']]
print(f"\n{'[FIMA-Q APPLICABLE]':-^90}")
print(f"  {len(can_fimaq)} layers (same as APB-applicable nn.Linear).")
print(f"  Mixed-precision: attn (qkv/proj) → W4A4, MLP (fc1/fc2) → W8A8")

print(f"\nNOTE: relative_position_bias_table là nn.Parameter (raw tensor), không phải Module.")
print(f"  → Tự động bị loại bởi apply_apb() và apply_fimaq() (scan chỉ xử lý nn.Module).")
print("="*92)


LAYER ANALYSIS — APB & FIMA-Q APPLICABILITY
Model: Swin-S (torchvision)  |  Source APB: rebuildapb.ipynb | Source FIMA-Q: ShiheWang/FIMA-Q

-------------------------------------[CAN APPLY APB]--------------------------------------
  Layer Name                                                 Type             Params
  -----------------------------------------------------------------------------------
  features.1.0.attn.qkv                                      Linear           27,936
  features.1.0.attn.proj                                     Linear            9,312
  features.1.0.mlp.0                                         Linear           37,248
  features.1.0.mlp.3                                         Linear           36,960
  features.1.1.attn.qkv                                      Linear           27,936
  features.1.1.attn.proj                                     Linear            9,312
  features.1.1.mlp.0                                         Linear           37,248
  f

In [14]:
# ============================================================================
# CELL 8: Training Configuration
# ============================================================================
# Pipeline summary:
#   Cell 9  : Fine-tune Baseline Swin-S on ImageNette (QAT-free)
#   Cell 10 : QAT-fine-tune APB Swin-S on ImageNette
#   Cell 11 : FIMA-Q PTQ calibration + Fisher computation (NO training loop)
#   Cell 12 : Create APB+FIMA-Q, then QAT-fine-tune on ImageNette
#   Cell 13 : 4-model evaluation
#
# FIMA-Q note:
#   FIMA-Q is Post-Training Quantization (PTQ).
#   It does NOT use a training loop — calibration on ~4 mini-batches (~128 samples).
#   No optimizer is created for FIMA-Q (quantization parameters are non-trainable buffers).
#   Reference: https://github.com/ShiheWang/FIMA-Q / test_quant.py
# ============================================================================

total_epochs  = QUICK_EPOCHS if QUICK_RUN else 20
learning_rate = 1e-4
weight_decay  = 0.01
freeze_epoch  = max(1, total_epochs // 2)   # APB: freeze alpha/delta after this epoch
max_grad_norm = 1.0

# ── Baseline optimizer ────────────────────────────────────────────────────
optimizer_baseline = optim.AdamW(model_baseline.parameters(),
                                 lr=learning_rate, weight_decay=weight_decay)
scheduler_baseline = optim.lr_scheduler.CosineAnnealingLR(optimizer_baseline,
                                                           T_max=total_epochs)

# ── APB optimizer ─────────────────────────────────────────────────────────
optimizer_apb = optim.AdamW(model_apb.parameters(),
                             lr=learning_rate, weight_decay=weight_decay)
scheduler_apb = optim.lr_scheduler.CosineAnnealingLR(optimizer_apb, T_max=total_epochs)

# ── FIMA-Q: PTQ — no optimizer (calibration-only, see Cell 11) ───────────
# APB+FIMA-Q optimizer (will be created in Cell 12 after model is built)
# Placeholder variables for result tracking:
optimizer_apb_fimaq = None
scheduler_apb_fimaq = None

criterion = nn.CrossEntropyLoss()

# ── Training state ────────────────────────────────────────────────────────
params_frozen          = False
best_acc_baseline      = 0.0
best_acc_apb           = 0.0
best_acc_fimaq         = 0.0   # accuracy after PTQ calibration (no training)
best_acc_apb_fimaq     = 0.0

results = {
    'baseline':  {'train_loss': [], 'val_loss': [], 'val_acc': []},
    'apb':       {'train_loss': [], 'val_loss': [], 'val_acc': [],
                  'binary_pct': [], 'compression_ratio': []},
    'fimaq':     {'val_acc': [], 'compression_ratio': []},
    'apb_fimaq': {'train_loss': [], 'val_loss': [], 'val_acc': [],
                  'binary_pct': [], 'compression_ratio': []},
}

save_dir = Path('./checkpoints')
save_dir.mkdir(exist_ok=True)
save_path_baseline   = save_dir / 'swin_s_baseline_best.pth'
save_path_apb        = save_dir / 'swin_s_apb_best.pth'
save_path_fimaq      = save_dir / 'swin_s_fimaq_best.pth'
save_path_apb_fimaq  = save_dir / 'swin_s_apb_fimaq_best.pth'
results_json_path    = save_dir / 'training_results.json'

apb_stats_history    = []   # per-epoch APB binarization stats for APB training loop

print("="*60)
print("TRAINING CONFIGURATION" + (" [QUICK_RUN]" if QUICK_RUN else " [FULL]"))
print("="*60)
print(f"  Epochs (baseline + APB + APB+FIMA-Q): {total_epochs}")
print(f"  Learning rate:  {learning_rate}")
print(f"  Weight decay:   {weight_decay}")
print(f"  APB freeze at:  epoch {freeze_epoch}")
print(f"  Gradient clip:  {max_grad_norm}")
print(f"  Device:         {device}")
print(f"\n  FIMA-Q: PTQ (no training) — calibration in Cell 11")
print("="*60)


TRAINING CONFIGURATION [QUICK_RUN]
  Epochs (baseline + APB + APB+FIMA-Q): 3
  Learning rate:  0.0001
  Weight decay:   0.01
  APB freeze at:  epoch 1
  Gradient clip:  1.0
  Device:         cpu

  FIMA-Q: PTQ (no training) — calibration in Cell 11


In [15]:
# ============================================================================
# CELL 9: Training Loop - Baseline Swin-S
# ============================================================================
import time as _time
import json

print("\n" + "="*60)
print("TRAINING BASELINE SWIN-S")
print("="*60 + "\n")

baseline_start = _time.time()
log_interval = max(1, len(train_loader) // 4)

for epoch in range(total_epochs):
    epoch_start = _time.time()
    model_baseline.train()
    running_loss = 0.0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_baseline.zero_grad()
        outputs = model_baseline(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_baseline.parameters(), max_grad_norm)
        optimizer_baseline.step()
        running_loss += loss.item()

        if (i + 1) % log_interval == 0:
            print(f"  Batch [{i+1:>3}/{len(train_loader)}] Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    val_res = evaluate(model_baseline, val_loader, criterion, device)

    results['baseline']['train_loss'].append(train_loss)
    results['baseline']['val_loss'].append(val_res['loss'])
    results['baseline']['val_acc'].append(val_res['accuracy'])

    epoch_time = _time.time() - epoch_start
    print(f"\nEpoch [{epoch+1:>2}/{total_epochs}] BASELINE | "
          f"Train: {train_loss:.4f}  Val: {val_res['loss']:.4f}  "
          f"Acc: {val_res['accuracy']:.2f}%  [{epoch_time:.0f}s]")

    if val_res['accuracy'] > best_acc_baseline:
        best_acc_baseline = val_res['accuracy']
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model_baseline.state_dict(),
            'optimizer_state_dict': optimizer_baseline.state_dict(),
            'accuracy': best_acc_baseline
        }, save_path_baseline)
        print(f"  → New best baseline: {best_acc_baseline:.2f}%  (saved)")

    scheduler_baseline.step()
    print()

baseline_train_time = _time.time() - baseline_start
print("="*60)
print(f"BASELINE DONE — Best Val Acc: {best_acc_baseline:.2f}%")
print(f"Training time: {baseline_train_time/60:.1f} min")
print("="*60)

# Save results dict to disk so Cell 12 can recover it even after a kernel reset
results_json_path = save_dir / 'training_results.json'
with open(results_json_path, 'w') as f:
    json.dump(results, f)
print(f" Training history saved → {results_json_path}")



TRAINING BASELINE SWIN-S

  Batch [ 12/50] Loss: 1.5563
  Batch [ 24/50] Loss: 0.3689
  Batch [ 36/50] Loss: 0.4273
  Batch [ 48/50] Loss: 0.1023

Epoch [ 1/3] BASELINE | Train: 1.0617  Val: 0.0674  Acc: 98.00%  [111s]
  → New best baseline: 98.00%  (saved)

  Batch [ 12/50] Loss: 0.0003
  Batch [ 24/50] Loss: 0.0034
  Batch [ 36/50] Loss: 0.0681
  Batch [ 48/50] Loss: 0.0060

Epoch [ 2/3] BASELINE | Train: 0.0618  Val: 0.0197  Acc: 99.00%  [118s]
  → New best baseline: 99.00%  (saved)

  Batch [ 12/50] Loss: 0.0009
  Batch [ 24/50] Loss: 0.0630
  Batch [ 36/50] Loss: 0.0002
  Batch [ 48/50] Loss: 0.0523

Epoch [ 3/3] BASELINE | Train: 0.0111  Val: 0.0273  Acc: 99.00%  [123s]

BASELINE DONE — Best Val Acc: 99.00%
Training time: 6.2 min
 Training history saved → checkpoints\training_results.json


In [16]:
# ============================================================================
# CELL 10: Training Loop - APB Swin-S
# ============================================================================
import json

print("\n" + "="*60)
print("TRAINING APB SWIN-S")
print("="*60 + "\n")

apb_start = _time.time()

for epoch in range(total_epochs):
    if epoch == freeze_epoch and not params_frozen:
        print("\n" + "="*40)
        print(f"EPOCH {epoch+1}: Freezing alpha and delta")
        print("="*40 + "\n")
        for module in model_apb.modules():
            if isinstance(module, APBLayer):
                module.alpha.requires_grad = False
                module.delta.requires_grad = False
        params_frozen = True

    model_apb.train()
    running_loss = 0.0

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_apb.zero_grad()
        outputs = model_apb(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_apb.parameters(), max_grad_norm)
        optimizer_apb.step()
        running_loss += loss.item()

        if (i + 1) % log_interval == 0:
            print(f"  Batch [{i+1:>3}/{len(train_loader)}] Loss: {loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    val_res = evaluate(model_apb, val_loader, criterion, device)

    results['apb']['train_loss'].append(train_loss)
    results['apb']['val_loss'].append(val_res['loss'])
    results['apb']['val_acc'].append(val_res['accuracy'])

    comp_stats = get_apb_compression_stats(model_apb)
    results['apb']['binary_pct'].append(comp_stats['binary_percentage'])
    results['apb']['compression_ratio'].append(comp_stats['compression_ratio'])
    apb_stats_history.append({'epoch': epoch + 1, **comp_stats})

    epoch_label = "[FROZEN]" if params_frozen else ""
    print(f"\nEpoch [{epoch+1:>2}/{total_epochs}] APB {epoch_label:8s}| "
          f"Train: {train_loss:.4f}  Val: {val_res['loss']:.4f}  "
          f"Acc: {val_res['accuracy']:.2f}%")
    print(f"  APB → binary: {comp_stats['binary_percentage']:.2f}%  "
          f"compression: {comp_stats['compression_ratio']:.2f}x")

    if val_res['accuracy'] > best_acc_apb:
        best_acc_apb = val_res['accuracy']
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model_apb.state_dict(),
            'optimizer_state_dict': optimizer_apb.state_dict(),
            'accuracy': best_acc_apb,
            'compression_stats': comp_stats
        }, save_path_apb)
        print(f"  → New best APB: {best_acc_apb:.2f}%  (saved)")

    scheduler_apb.step()
    print()

apb_train_time = _time.time() - apb_start
print("="*60)
print(f"APB TRAINING DONE — Best Val Acc: {best_acc_apb:.2f}%")
print(f"Training time: {apb_train_time/60:.1f} min")
print("="*60)

# Merge APB results into saved JSON
results_json_path = save_dir / 'training_results.json'
try:
    with open(results_json_path) as f:
        saved = json.load(f)
    saved['apb'] = results['apb']
except Exception:
    saved = results
with open(results_json_path, 'w') as f:
    json.dump(saved, f)
print(f" Full training history saved → {results_json_path}")



TRAINING APB SWIN-S

  Batch [ 12/50] Loss: 2.7640
  Batch [ 24/50] Loss: 2.3079
  Batch [ 36/50] Loss: 2.5513
  Batch [ 48/50] Loss: 2.3337

Epoch [ 1/3] APB         | Train: 2.4422  Val: 2.3105  Acc: 18.00%
  APB → binary: 99.91%  compression: 31.14x
  → New best APB: 18.00%  (saved)


EPOCH 2: Freezing alpha and delta

  Batch [ 12/50] Loss: 1.9532
  Batch [ 24/50] Loss: 2.4712
  Batch [ 36/50] Loss: 2.3371
  Batch [ 48/50] Loss: 2.2514

Epoch [ 2/3] APB [FROZEN]| Train: 2.2555  Val: 2.2096  Acc: 22.00%
  APB → binary: 99.91%  compression: 31.14x
  → New best APB: 22.00%  (saved)

  Batch [ 12/50] Loss: 2.1149
  Batch [ 24/50] Loss: 2.2671
  Batch [ 36/50] Loss: 1.5633
  Batch [ 48/50] Loss: 2.1238

Epoch [ 3/3] APB [FROZEN]| Train: 2.1500  Val: 2.0253  Acc: 25.00%
  APB → binary: 99.91%  compression: 31.14x
  → New best APB: 25.00%  (saved)

APB TRAINING DONE — Best Val Acc: 25.00%
Training time: 15.1 min
 Full training history saved → checkpoints\training_results.json


In [ ]:
# Cell 11: FIMA-Q PTQ Calibration + Fisher Computation
# PTQ: không dùng training loop — calibrate ~4 batches (~128 samples)
# Workflow:
#   1. Load baseline checkpoint vào model_fimaq
#   2. calibrate_fimaq(): thu thập min/max → set scale/zero_point → mode='quant_forward'
#   3. compute_fisher_for_model(): tính Fisher scores để dùng cho APB+FIMA-Q
#   4. Evaluate FIMA-Q trên ImageNette val
#   5. Save checkpoint

import copy as _copy
import json as _json

print("="*60)
print("CELL 11: FIMA-Q PTQ Calibration")
print("="*60)

# Step 1: Load best baseline weights vào model_fimaq
print("\nStep 1 — Loading baseline checkpoint into FIMA-Q model...")
if save_path_baseline.exists():
    ckpt_base = torch.load(save_path_baseline, map_location=device, weights_only=False)
    fimaq_state = model_fimaq.state_dict()
    base_state  = ckpt_base['model_state_dict']
    loaded = skipped = 0
    for k in fimaq_state:
        if k in base_state and fimaq_state[k].shape == base_state[k].shape:
            fimaq_state[k] = base_state[k]
            loaded += 1
        else:
            skipped += 1
    model_fimaq.load_state_dict(fimaq_state)
    print(f"  Loaded {loaded} tensors  ({skipped} skipped/buffer)")
    print(f"  Baseline best acc: {ckpt_base['accuracy']:.2f}%")
else:
    print("  WARNING: baseline checkpoint not found — using pretrained weights")

for _, m in model_fimaq.named_modules():
    if isinstance(m, FIMAQLinear):
        m.mode = 'calibration'
        m.raw_input = None
        m.calibrated = False

# Step 2: Calibrate FIMA-Q
print("\nStep 2 — PTQ calibration (min-max)...")
n_calib = 2 if QUICK_RUN else 4
model_fimaq = calibrate_fimaq(model_fimaq, val_loader, device, n_batches=n_calib)
fimaq_quant_stats = get_fimaq_compression_stats(model_fimaq)
print(f"\n  Compression stats (W+A):")
print(f"    4-bit layers: {fimaq_quant_stats['layers_4bit']}")
print(f"    8-bit layers: {fimaq_quant_stats['layers_8bit']}")
print(f"    CR weight:    {fimaq_quant_stats['cr_weight']:.2f}x")
print(f"    CR act:       {fimaq_quant_stats['cr_activation']:.2f}x")
print(f"    CR combined:  {fimaq_quant_stats['compression_ratio']:.2f}x")

# Step 3: Compute Fisher scores
print("\nStep 3 — Computing Fisher Information scores...")
fisher_scores = compute_fisher_for_model(
    model_fimaq, val_loader, device, criterion, n_batches=n_calib
)

# Step 4: Evaluate
print("\nStep 4 — Evaluating FIMA-Q on ImageNette val...")
fimaq_val_res = evaluate(model_fimaq, val_loader, criterion, device, measure_speed=True)
best_acc_fimaq = fimaq_val_res['accuracy']
results['fimaq']['val_acc'].append(best_acc_fimaq)
results['fimaq']['compression_ratio'].append(fimaq_quant_stats['compression_ratio'])
print(f"\n  Val Accuracy: {fimaq_val_res['accuracy']:.2f}%")
print(f"  Latency:      {fimaq_val_res['avg_inference_time']*1000:.2f} ms/batch")
print(f"  Throughput:   {fimaq_val_res['throughput']:.1f} img/s")

# Step 5: Save checkpoint
print("\nStep 5 — Saving...")
torch.save({
    'model_state_dict': model_fimaq.state_dict(),
    'accuracy': best_acc_fimaq,
    'compression_stats': fimaq_quant_stats,
    'fisher_scores': {k: v.cpu() for k, v in fisher_scores.items()},
}, save_path_fimaq)
print(f"  Saved → {save_path_fimaq}")

try:
    with open(results_json_path) as f:
        saved_r = _json.load(f)
except Exception:
    saved_r = dict(results)
saved_r['fimaq'] = results['fimaq']
with open(results_json_path, 'w') as f:
    _json.dump(saved_r, f, indent=2)

print("\n" + "="*60)
print(f"FIMA-Q PTQ DONE — Val Acc: {best_acc_fimaq:.2f}%")
print("  fisher_scores ready for apply_apb_fimaq()")
print("="*60)


CELL 11: FIMA-Q PTQ Calibration

Step 1 — Loading best baseline checkpoint into FIMA-Q model...
  Loaded 353 tensors from baseline checkpoint  (297 skipped/buffer)
  Baseline best acc: 99.00%

Step 2 — Running PTQ calibration (min-max, ~4 batches)...
  Collected calibration data from 2 batch(es)
  Calibrated 51 FIMAQLinear layer(s)

  Compression stats (Weight + Activation):
    4-bit layers:        48
    8-bit layers:        51
    Weight CR:           4.77x
    Activation CR:       4.64x
    Combined CR:         4.77x

Step 3 — Computing Fisher Information scores...
  Computed Fisher scores for 99 FIMAQLinear layer(s)

Step 4 — Evaluating FIMA-Q on ImageNette val...

  FIMA-Q Val Accuracy:  99.00%
  Avg batch latency:    1079.77 ms
  Throughput:           3.7 img/s

Step 5 — Saving FIMA-Q checkpoint...
  Saved -> checkpoints\swin_s_fimaq_best.pth

FIMA-Q PTQ DONE — Val Acc: 99.00%
  No training loop used — pure post-training quantization.
  fisher_scores ready for apply_apb_fimaq() 

In [ ]:
# Cell 12: APB + FIMA-Q — Tạo combined model và fine-tune (QAT)
#
# Workflow:
#   1. Copy model_fimaq (calibrated, quant_forward mode)
#   2. apply_apb_fimaq(): thay FIMAQLinear → APBFIMAQLayer
#      - Fisher scores xác định weights được giữ FP32 (top 10%)
#      - Phần còn lại: binarizable bởi APB threshold
#      - Activation quantization từ FIMA-Q được giữ nguyên
#   3. Fine-tune với QAT: APB binarize weights, FIMA-Q quantize activations

import copy as _copy
import json as _json

print("="*60)
print("CELL 12: APB + FIMA-Q Combined Model")
print("="*60)

# Step 1: Build APB+FIMA-Q
print("\nStep 1 — Creating APB+FIMA-Q model...")
model_apb_fimaq = _copy.deepcopy(model_fimaq)
model_apb_fimaq = apply_apb_fimaq(
    model_apb_fimaq,
    fisher_scores     = fisher_scores,
    fisher_keep_ratio = 0.10,
    a_bits            = 8,
)
model_apb_fimaq.to(device)

apb_fimaq_total, apb_fimaq_trainable = count_parameters(model_apb_fimaq)
apb_fimaq_size = get_model_size(model_apb_fimaq)
print(f"\n  {apb_fimaq_total:,} params  |  {apb_fimaq_size:.1f} MB")
apb_fimaq_init_stats = get_apb_fimaq_compression_stats(model_apb_fimaq)
print(f"  Initial binary pct:  {apb_fimaq_init_stats.get('binary_pct', 0):.1f}%")
print(f"  Fisher-protected:    top 10% Fisher weights → FP32")
print(f"  Activation quant:    8-bit asymmetric (từ FIMA-Q calibration)")

# Step 2: Optimizer + scheduler
optimizer_apb_fimaq = optim.AdamW(model_apb_fimaq.parameters(),
                                   lr=learning_rate, weight_decay=weight_decay)
scheduler_apb_fimaq = optim.lr_scheduler.CosineAnnealingLR(optimizer_apb_fimaq,
                                                             T_max=total_epochs)

# Step 3: QAT fine-tuning
print("\nStep 2 — QAT fine-tuning APB+FIMA-Q...")
apb_fimaq_frozen = False
apb_fimaq_start  = _time.time()

for epoch in range(total_epochs):
    if epoch == freeze_epoch and not apb_fimaq_frozen:
        print(f"\n  Epoch {epoch+1}: Freezing APB alpha/delta")
        for m in model_apb_fimaq.modules():
            if isinstance(m, APBFIMAQLayer):
                m.alpha.requires_grad_(False)
                m.delta.requires_grad_(False)
        apb_fimaq_frozen = True

    model_apb_fimaq.train()
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_apb_fimaq.zero_grad()
        loss = criterion(model_apb_fimaq(inputs), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_apb_fimaq.parameters(), max_grad_norm)
        optimizer_apb_fimaq.step()
        running_loss += loss.item()
        if (i + 1) % max(1, len(train_loader) // 4) == 0:
            print(f"    Batch [{i+1:>3}/{len(train_loader)}] Loss: {loss.item():.4f}")

    train_loss   = running_loss / len(train_loader)
    val_res      = evaluate(model_apb_fimaq, val_loader, criterion, device)
    comp_st      = get_apb_fimaq_compression_stats(model_apb_fimaq)

    results['apb_fimaq']['train_loss'].append(train_loss)
    results['apb_fimaq']['val_loss'].append(val_res['loss'])
    results['apb_fimaq']['val_acc'].append(val_res['accuracy'])
    results['apb_fimaq']['binary_pct'].append(comp_st.get('binary_pct', 0))
    results['apb_fimaq']['compression_ratio'].append(comp_st.get('cr_weight', 1.0))

    frozen_tag = " [FROZEN]" if apb_fimaq_frozen else ""
    print(f"\n  Epoch [{epoch+1:>2}/{total_epochs}] APB+FIMA-Q{frozen_tag:8s}| "
          f"Train: {train_loss:.4f}  Val: {val_res['loss']:.4f}  "
          f"Acc: {val_res['accuracy']:.2f}%")
    print(f"    binary: {comp_st.get('binary_pct', 0):.2f}%  "
          f"cr_weight: {comp_st.get('cr_weight', 1.0):.2f}x  "
          f"cr_act: {comp_st.get('cr_activation', 1.0):.2f}x")

    if val_res['accuracy'] > best_acc_apb_fimaq:
        best_acc_apb_fimaq = val_res['accuracy']
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model_apb_fimaq.state_dict(),
            'accuracy': best_acc_apb_fimaq,
            'compression_stats': comp_st,
        }, save_path_apb_fimaq)
        print(f"    → New best APB+FIMA-Q: {best_acc_apb_fimaq:.2f}%  (saved)")

    scheduler_apb_fimaq.step()
    print()

apb_fimaq_time = _time.time() - apb_fimaq_start

try:
    with open(results_json_path) as f:
        saved_r = _json.load(f)
except Exception:
    saved_r = dict(results)
saved_r['apb_fimaq'] = results['apb_fimaq']
with open(results_json_path, 'w') as f:
    _json.dump(saved_r, f, indent=2)

print("="*60)
print(f"APB+FIMA-Q DONE — Best Val Acc: {best_acc_apb_fimaq:.2f}%")
print(f"Training time: {apb_fimaq_time/60:.1f} min")
print("="*60)


CELL 12: APB + FIMA-Q Combined Model

Step 1 — Creating APB+FIMA-Q model from FIMA-Q (calibrated)...
  ✓ APBFIMAQLayer: features.1.0.attn.qkv  (fp_mask=2,765 protected weights)
  ✓ APBFIMAQLayer: features.1.0.attn.proj  (fp_mask=922 protected weights)
  ✓ APBFIMAQLayer: features.1.0.mlp.0  (fp_mask=3,687 protected weights)
  ✓ APBFIMAQLayer: features.1.0.mlp.3  (fp_mask=3,687 protected weights)
  ✓ APBFIMAQLayer: features.1.1.attn.qkv  (fp_mask=2,765 protected weights)
  ✓ APBFIMAQLayer: features.1.1.attn.proj  (fp_mask=922 protected weights)
  ✓ APBFIMAQLayer: features.1.1.mlp.0  (fp_mask=3,687 protected weights)
  ✓ APBFIMAQLayer: features.1.1.mlp.3  (fp_mask=3,687 protected weights)
  ✓ APBFIMAQLayer: features.2.reduction  (fp_mask=7,373 protected weights)
  ✓ APBFIMAQLayer: features.3.0.attn.qkv  (fp_mask=11,060 protected weights)
  ✓ APBFIMAQLayer: features.3.0.attn.proj  (fp_mask=3,687 protected weights)
  ✓ APBFIMAQLayer: features.3.0.mlp.0  (fp_mask=14,746 protected weights)
  

In [19]:
# Cell 13: Final Evaluation — tất cả models
# So sánh: Pretrained (model gốc) / Baseline / APB / FIMA-Q / APB+FIMA-Q
# Metrics: accuracy, FLOPs, model size (MB), latency, throughput, compression ratio

import os, json as _json

print("\n" + "="*80)
print("FINAL EVALUATION — ALL MODELS ON IMAGENETTE")
print("="*80)

# ── Load best checkpoints ─────────────────────────────────────────────────
print("  Pretrained:  using live model (no fine-tuning)")

ckpt_baseline = torch.load(save_path_baseline, map_location=device, weights_only=False)
model_baseline.load_state_dict(ckpt_baseline['model_state_dict'])
print(f"  Baseline:    loaded  (ep {ckpt_baseline['epoch']}, acc {ckpt_baseline['accuracy']:.2f}%)")

ckpt_apb = torch.load(save_path_apb, map_location=device, weights_only=False)
model_apb.load_state_dict(ckpt_apb['model_state_dict'])
print(f"  APB:         loaded  (ep {ckpt_apb['epoch']}, acc {ckpt_apb['accuracy']:.2f}%)")

if save_path_fimaq.exists():
    ckpt_fimaq = torch.load(save_path_fimaq, map_location=device, weights_only=False)
    fimaq_sd = model_fimaq.state_dict()
    for k, v in ckpt_fimaq['model_state_dict'].items():
        if k in fimaq_sd and fimaq_sd[k].shape == v.shape:
            fimaq_sd[k] = v
    model_fimaq.load_state_dict(fimaq_sd)
    print(f"  FIMA-Q:      loaded  (acc {ckpt_fimaq['accuracy']:.2f}%)")
else:
    print("  FIMA-Q:      (using in-memory calibrated model)")

if save_path_apb_fimaq.exists():
    ckpt_apb_fimaq = torch.load(save_path_apb_fimaq, map_location=device, weights_only=False)
    model_apb_fimaq.load_state_dict(ckpt_apb_fimaq['model_state_dict'])
    print(f"  APB+FIMA-Q:  loaded  (ep {ckpt_apb_fimaq['epoch']}, acc {ckpt_apb_fimaq['accuracy']:.2f}%)")
else:
    print("  APB+FIMA-Q:  (using in-memory model)")

# ── Speed benchmark ───────────────────────────────────────────────────────
print("\nRunning inference benchmark...")
pretrained_res  = evaluate(model_pretrained, val_loader, criterion, device, measure_speed=True)
baseline_res    = evaluate(model_baseline,   val_loader, criterion, device, measure_speed=True)
apb_res         = evaluate(model_apb,        val_loader, criterion, device, measure_speed=True)
fimaq_res       = evaluate(model_fimaq,      val_loader, criterion, device, measure_speed=True)
apb_fimaq_res   = evaluate(model_apb_fimaq,  val_loader, criterion, device, measure_speed=True)

# ── FLOPs ─────────────────────────────────────────────────────────────────
print("Measuring FLOPs...")
_, flops_pretrained = get_flops(model_pretrained, device=device)
_, flops_baseline   = get_flops(model_baseline,   device=device)
_, flops_apb        = get_flops(model_apb,         device=device)
_, flops_fimaq      = get_flops(model_fimaq,       device=device)
_, flops_apb_fimaq  = get_flops(model_apb_fimaq,   device=device)

# ── Compression stats ─────────────────────────────────────────────────────
comp_apb       = get_apb_compression_stats(model_apb)
comp_fimaq     = get_fimaq_compression_stats(model_fimaq)
comp_apb_fimaq = get_apb_fimaq_compression_stats(model_apb_fimaq)

# ── File sizes ────────────────────────────────────────────────────────────
sz_pretrained = get_model_size(model_pretrained)
sz_base  = os.path.getsize(save_path_baseline)  / 1024 / 1024 if save_path_baseline.exists()  else 0
sz_apb   = os.path.getsize(save_path_apb)       / 1024 / 1024 if save_path_apb.exists()       else 0
sz_fimaq = os.path.getsize(save_path_fimaq)     / 1024 / 1024 if save_path_fimaq.exists()     else 0
sz_afq   = os.path.getsize(save_path_apb_fimaq) / 1024 / 1024 if save_path_apb_fimaq.exists() else 0

# ── Comparison table ──────────────────────────────────────────────────────
print("\n" + "="*110)
print("COMPARISON TABLE — SWIN-S ON IMAGENETTE (10 classes)")
print("  (*) Model gốc: ImageNet-1K backbone, head random 10-class chưa train")
print("="*110)

def row(metric, p, b, a, f, af):
    return f"  {metric:<28} {str(p):<16} {str(b):<16} {str(a):<16} {str(f):<16} {str(af)}"

print(row("Metric", "Pretrained(*)", "Baseline", "APB", "FIMA-Q", "APB+FIMA-Q"))
print("  " + "-"*108)
print(row("Accuracy (%)",
    f"{pretrained_res['accuracy']:.2f}(*)",
    f"{baseline_res['accuracy']:.2f}",
    f"{apb_res['accuracy']:.2f}",
    f"{fimaq_res['accuracy']:.2f}",
    f"{apb_fimaq_res['accuracy']:.2f}"))
print(row("FLOPs", flops_pretrained, flops_baseline, flops_apb, flops_fimaq, flops_apb_fimaq))
print(row("Model size (MB)",
    f"{sz_pretrained:.1f}", f"{sz_base:.1f}", f"{sz_apb:.1f}", f"{sz_fimaq:.1f}", f"{sz_afq:.1f}"))
print(row("Avg latency (ms/batch)",
    f"{pretrained_res['avg_inference_time']*1e3:.1f}",
    f"{baseline_res['avg_inference_time']*1e3:.1f}",
    f"{apb_res['avg_inference_time']*1e3:.1f}",
    f"{fimaq_res['avg_inference_time']*1e3:.1f}",
    f"{apb_fimaq_res['avg_inference_time']*1e3:.1f}"))
print(row("Throughput (img/s)",
    f"{pretrained_res['throughput']:.1f}",
    f"{baseline_res['throughput']:.1f}",
    f"{apb_res['throughput']:.1f}",
    f"{fimaq_res['throughput']:.1f}",
    f"{apb_fimaq_res['throughput']:.1f}"))
print(row("Speedup vs baseline",
    f"{baseline_res['avg_inference_time']/pretrained_res['avg_inference_time']:.2f}x",
    "1.00x",
    f"{baseline_res['avg_inference_time']/apb_res['avg_inference_time']:.2f}x",
    f"{baseline_res['avg_inference_time']/fimaq_res['avg_inference_time']:.2f}x",
    f"{baseline_res['avg_inference_time']/apb_fimaq_res['avg_inference_time']:.2f}x"))
print(row("CR weight (x)",
    "1.00", "1.00",
    f"{comp_apb['compression_ratio']:.2f}",
    f"{comp_fimaq['cr_weight']:.2f}",
    f"{comp_apb_fimaq.get('cr_weight', 1.0):.2f}"))
print(row("CR activation (x)",
    "1.00", "1.00", "1.00 (none)",
    f"{comp_fimaq['cr_activation']:.2f}",
    f"{comp_apb_fimaq.get('cr_activation', 1.0):.2f}"))
print(row("CR combined (W+A) (x)",
    "1.00", "1.00", "-",
    f"{comp_fimaq['compression_ratio']:.2f}",
    f"{comp_apb_fimaq.get('cr_weight', 1.0):.2f}"))
print(row("Binary weight % (APB)",
    "-", "-",
    f"{comp_apb['binary_percentage']:.1f}%",
    "-",
    f"{comp_apb_fimaq.get('binary_pct', 0):.1f}%"))
print("="*110)
print("  (*) Accuracy thấp vì head random — đây là lower-bound reference trước khi adapt.")
print()

# ── Save results ──────────────────────────────────────────────────────────
final_results = {
    'pretrained': {'accuracy': pretrained_res['accuracy'], 'flops': flops_pretrained,
                   'latency_ms': round(pretrained_res['avg_inference_time']*1e3, 2),
                   'throughput': round(pretrained_res['throughput'], 1),
                   'note': 'model_goc_no_adaptation'},
    'baseline':   {'accuracy': baseline_res['accuracy'],  'flops': flops_baseline,
                   'latency_ms': round(baseline_res['avg_inference_time']*1e3, 2),
                   'throughput': round(baseline_res['throughput'], 1),
                   'checkpoint_mb': round(sz_base, 1)},
    'apb':        {'accuracy': apb_res['accuracy'],       'flops': flops_apb,
                   'latency_ms': round(apb_res['avg_inference_time']*1e3, 2),
                   'throughput': round(apb_res['throughput'], 1),
                   'cr_weight': comp_apb['compression_ratio'],
                   'binary_pct': comp_apb['binary_percentage'],
                   'checkpoint_mb': round(sz_apb, 1)},
    'fimaq':      {'accuracy': fimaq_res['accuracy'],     'flops': flops_fimaq,
                   'latency_ms': round(fimaq_res['avg_inference_time']*1e3, 2),
                   'throughput': round(fimaq_res['throughput'], 1),
                   'cr_weight': comp_fimaq['cr_weight'],
                   'cr_activation': comp_fimaq['cr_activation'],
                   'cr_combined': comp_fimaq['compression_ratio'],
                   'checkpoint_mb': round(sz_fimaq, 1)},
    'apb_fimaq':  {'accuracy': apb_fimaq_res['accuracy'], 'flops': flops_apb_fimaq,
                   'latency_ms': round(apb_fimaq_res['avg_inference_time']*1e3, 2),
                   'throughput': round(apb_fimaq_res['throughput'], 1),
                   'cr_weight': comp_apb_fimaq.get('cr_weight', 1.0),
                   'cr_activation': comp_apb_fimaq.get('cr_activation', 1.0),
                   'binary_pct': comp_apb_fimaq.get('binary_pct', 0),
                   'checkpoint_mb': round(sz_afq, 1)},
}
summary_json = save_dir / 'final_results_all_models.json'
with open(summary_json, 'w') as f:
    _json.dump(final_results, f, indent=2)
print(f"  Saved → {summary_json}")

final_eval_res     = final_results
baseline_res_eval  = baseline_res
apb_res_eval       = apb_res
fimaq_res_eval     = fimaq_res
apb_fimaq_res_eval = apb_fimaq_res

print("\n" + "="*80)
print("EVALUATION COMPLETE!")
print("="*80)



FINAL EVALUATION — ALL 5 MODELS ON IMAGENETTE
  Pretrained:  using live model (no fine-tuning, no checkpoint)
  Baseline:    loaded  (ep 2, acc 99.00%)
  APB:         loaded  (ep 3, acc 25.00%)
  FIMA-Q:      loaded  (acc 99.00%)
  APB+FIMA-Q:  loaded  (ep 3, acc 54.00%)

Running inference speed benchmark (val_loader)...

Measuring FLOPs (thop)...

COMPARISON TABLE — SWIN-S ON IMAGENETTE (10 classes)
  Pretrained = model goc (ImageNet-1K weights, no ImageNette adaptation, random 10-class head)
  Metric                       Pretrained(*)    Baseline         APB              FIMA-Q           APB+FIMA-Q
  ------------------------------------------------------------------------------------------------------------
  Accuracy (%)                 8.00(*)          99.00            25.00            99.00            54.00
  FLOPs                        1                1                7                7                7
  Model size (MB)              186.8            559.8            440.0   

In [ ]:
# ============================================================================
# CELL 14: Visualization — Training Curves + 4-Model Comparison
# ============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import json as _json

# ── Restore training history if lost ──────────────────────────────────────
if results_json_path.exists() and len(results['baseline']['train_loss']) == 0:
    with open(results_json_path) as f:
        results.update(_json.load(f))
    print(f"  Restored training history from {results_json_path}")

n_base = len(results['baseline']['train_loss'])
n_apb  = len(results['apb']['train_loss'])
n_afq  = len(results['apb_fimaq']['train_loss'])
print(f"  Baseline epochs: {n_base}  |  APB epochs: {n_apb}  |  APB+FIMA-Q epochs: {n_afq}")

base_range = list(range(1, n_base + 1))
apb_range  = list(range(1, n_apb  + 1))
afq_range  = list(range(1, n_afq  + 1))

COLORS = {'baseline': '#2196F3', 'apb': '#F44336',
          'fimaq': '#4CAF50',    'apb_fimaq': '#FF9800'}

# ── Figure 1: Training curves (6-panel) ───────────────────────────────────
fig1 = plt.figure(figsize=(18, 10))
fig1.suptitle("Swin-S Training Curves — Baseline / APB / APB+FIMA-Q",
              fontsize=15, fontweight='bold')
gs   = gridspec.GridSpec(2, 3, figure=fig1, hspace=0.45, wspace=0.35)

def plot_curve(ax, title, ylabel, key, add_fimaq=False):
    if n_base: ax.plot(base_range, results['baseline'][key],
                       'o-', color=COLORS['baseline'], label='Baseline', ms=4, lw=1.5)
    if n_apb:  ax.plot(apb_range,  results['apb'][key],
                       's-', color=COLORS['apb'],      label='APB',      ms=4, lw=1.5)
    if n_afq:  ax.plot(afq_range,  results['apb_fimaq'][key],
                       '^-', color=COLORS['apb_fimaq'],label='APB+FIMA-Q',ms=4, lw=1.5)
    ax.axvline(x=freeze_epoch, color='gray', ls='--', alpha=0.5, lw=1, label=f'Freeze ep{freeze_epoch}')
    ax.set_title(title, fontsize=11); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plot_curve(fig1.add_subplot(gs[0,0]), 'Training Loss',        'Loss',     'train_loss')
plot_curve(fig1.add_subplot(gs[0,1]), 'Validation Loss',      'Loss',     'val_loss')
plot_curve(fig1.add_subplot(gs[0,2]), 'Validation Accuracy (%)', 'Acc (%)', 'val_acc')
plot_curve(fig1.add_subplot(gs[1,0]), 'APB / APB+FIMA-Q: Binary Weight %', 'Binary (%)', 'binary_pct')
plot_curve(fig1.add_subplot(gs[1,1]), 'APB / APB+FIMA-Q: Weight CR (x)',   'CR',         'compression_ratio')

# FIMA-Q annotation (single PTQ point, no training curve)
ax6 = fig1.add_subplot(gs[1,2])
models_acc = {
    'Baseline': baseline_res_eval['accuracy'],
    'APB': apb_res_eval['accuracy'],
    'FIMA-Q': fimaq_res_eval['accuracy'],
    'APB+\nFIMA-Q': apb_fimaq_res_eval['accuracy'],
}
bar_colors = [COLORS['baseline'], COLORS['apb'], COLORS['fimaq'], COLORS['apb_fimaq']]
bars = ax6.bar(models_acc.keys(), models_acc.values(), color=bar_colors, alpha=0.85)
ax6.set_title('Final Val Accuracy — 4 Models', fontsize=11)
ax6.set_ylabel('Accuracy (%)'); ax6.set_ylim(0, 110)
for b in bars:
    ax6.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
             f'{b.get_height():.2f}%', ha='center', va='bottom', fontsize=9)
ax6.grid(True, alpha=0.3, axis='y')

chart1 = save_dir / 'swin_s_training_curves_4model.png'
fig1.savefig(chart1, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n  Chart 1 saved -> {chart1}")

# ── Figure 2: Compression vs. Accuracy trade-off ─────────────────────────
fig2, axes = plt.subplots(1, 3, figsize=(15, 5))
fig2.suptitle("Swin-S: Compression-Accuracy Trade-off (4 Models)", fontsize=14)

def get_cr_weight(name, res):
    if name == 'baseline': return 1.0
    if name == 'apb':      return res.get('cr_weight', 1.0)
    if name == 'fimaq':    return res.get('cr_weight', 1.0)
    if name == 'apb_fimaq': return res.get('cr_weight', 1.0)
    return 1.0

data4 = [
    ('Baseline',    final_eval_res['baseline']['accuracy'],  1.0, 1.0,
     baseline_res_eval['avg_inference_time']*1e3),
    ('APB',         final_eval_res['apb']['accuracy'],
     final_eval_res['apb']['cr_weight'],        1.0,
     apb_res_eval['avg_inference_time']*1e3),
    ('FIMA-Q',      final_eval_res['fimaq']['accuracy'],
     final_eval_res['fimaq']['cr_weight'],
     final_eval_res['fimaq']['cr_activation'],
     fimaq_res_eval['avg_inference_time']*1e3),
    ('APB+FIMA-Q',  final_eval_res['apb_fimaq']['accuracy'],
     final_eval_res['apb_fimaq']['cr_weight'],
     final_eval_res['apb_fimaq']['cr_activation'],
     apb_fimaq_res_eval['avg_inference_time']*1e3),
]
names_4  = [d[0] for d in data4]
accs_4   = [d[1] for d in data4]
cr_w_4   = [d[2] for d in data4]
cr_a_4   = [d[3] for d in data4]
lats_4   = [d[4] for d in data4]
cols_4   = [COLORS[k] for k in ['baseline','apb','fimaq','apb_fimaq']]

for ax, xvals, xlabel, title in [
    (axes[0], cr_w_4,  'Weight CR (x)',     'Accuracy vs Weight Compression'),
    (axes[1], cr_a_4,  'Activation CR (x)', 'Accuracy vs Activation Compression'),
    (axes[2], lats_4,  'Latency (ms/batch)','Accuracy vs Latency'),
]:
    sc = ax.scatter(xvals, accs_4, c=cols_4, s=180, zorder=5, edgecolors='black', lw=0.8)
    for i, n in enumerate(names_4):
        ax.annotate(n, (xvals[i], accs_4[i]),
                    textcoords="offset points", xytext=(6, 4), fontsize=9)
    ax.set_xlabel(xlabel); ax.set_ylabel('Accuracy (%)'); ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)

chart2 = save_dir / 'swin_s_compression_tradeoff.png'
fig2.tight_layout()
fig2.savefig(chart2, dpi=150, bbox_inches='tight')
plt.show()
print(f"  Chart 2 saved -> {chart2}")


  Baseline epochs: 3  |  APB epochs: 3

 Chart saved → checkpoints\swin_s_training_curves.png


C:\Users\user\AppData\Local\Temp\ipykernel_26564\3335539209.py:86: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Khôi phục training history từ JSON nếu mất biến sau kernel restart
import json

if results_json_path.exists() and len(results['baseline']['train_loss']) == 0:
    with open(results_json_path) as f:
        saved = json.load(f)
    results.update(saved)
    print("Training history restored from JSON.")
else:
    print("Training history OK.")

if 'ckpt_baseline' in dir():
    best_acc_baseline = float(ckpt_baseline['accuracy'])
if 'ckpt_apb' in dir():
    best_acc_apb = float(ckpt_apb['accuracy'])

n_base     = len(results['baseline']['train_loss'])
n_apb      = len(results['apb']['train_loss'])
n_afq      = len(results['apb_fimaq']['train_loss'])
base_range = list(range(1, n_base + 1))
apb_range  = list(range(1, n_apb  + 1))
afq_range  = list(range(1, n_afq  + 1))

print(f"Baseline: {n_base} epochs | APB: {n_apb} epochs | APB+FIMA-Q: {n_afq} epochs")


 Baseline history already present

  Baseline epochs: 3  (train_loss=[1.3030130869150163, 0.0423002084219479, 0.0060022456246224465])
  APB epochs:      3  (train_loss=[2.4172895669937136, 2.1816996264457704, 2.1700595116615293])
  best_acc_baseline: 99.00%
  best_acc_apb:      16.00%

→ Now re-run Cell 12 (visualization) to generate the complete chart!


In [ ]:

# ============================================================================
# CELL 13: Per-Layer APB Analysis
# ============================================================================
print("="*80)
print("PER-LAYER APB ANALYSIS")
print("="*80)

layer_stats = []
for name, module in model_apb.named_modules():
    if isinstance(module, APBLayer):
        s = module.get_stats()
        n_params = module.latent_weight.numel()
        layer_stats.append({
            'name': name,
            'n_params': n_params,
            'alpha': s['alpha'],
            'delta': s['delta'],
            'binary_pct': s['percent_binary'],
        })

# Print table
print(f"\n{'Layer Name':<55} {'Params':>10} {'Alpha':>8} {'Delta':>8} {'Binary%':>9}")
print("-"*92)
total_params_apb = 0
total_binary_params = 0
for s in layer_stats:
    binary_count = int(s['binary_pct'] / 100 * s['n_params'])
    total_params_apb += s['n_params']
    total_binary_params += binary_count
    print(f"  {s['name']:<53} {s['n_params']:>10,} {s['alpha']:>8.4f} {s['delta']:>8.4f} {s['binary_pct']:>8.2f}%")

print("-"*92)
overall_binary = (total_binary_params / total_params_apb * 100) if total_params_apb > 0 else 0
print(f"  {'TOTAL APB layers':<53} {total_params_apb:>10,} {'':>8} {'':>8} {overall_binary:>8.2f}%")
print(f"\n  APB applied to {len(layer_stats)} layers")
print(f"  Overall binarization: {total_binary_params:,} / {total_params_apb:,} = {overall_binary:.2f}%")

# Top 10 most binarized layers
print("\n Top 10 most binarized layers:")
sorted_layers = sorted(layer_stats, key=lambda x: x['binary_pct'], reverse=True)[:10]
for rank, s in enumerate(sorted_layers, 1):
    print(f"  {rank:>2}. {s['name']:<55} {s['binary_pct']:.2f}%")

# Bar chart — per-layer binary % (supports any number of layers)
n_layers = len(layer_stats)
fig_w = max(16, n_layers * 0.22)
fig2, ax = plt.subplots(figsize=(fig_w, 5))
names = []
for s in layer_stats:
    parts = s['name'].split('.')
    short = '.'.join(parts[-2:]) if len(parts) >= 2 else s['name']
    names.append(short)
bpcts = [s['binary_pct'] for s in layer_stats]
colors = ['#2ecc71' if b > 90 else '#f39c12' if b > 50 else '#e74c3c' for b in bpcts]
ax.bar(range(len(names)), bpcts, color=colors, alpha=0.85)
ax.axhline(y=90, color='blue',  linestyle='--', alpha=0.5, linewidth=1, label='90% (target)')
ax.axhline(y=50, color='gray',  linestyle=':',  alpha=0.5, linewidth=1, label='50%')
ax.set_xticks(range(len(names)))
fontsize = max(4, min(8, int(180 / n_layers)))
ax.set_xticklabels(names, rotation=90, fontsize=fontsize)
ax.set_title(f'Per-Layer Binarization Rate (%) — {n_layers} APB Layers', fontsize=13)
ax.set_ylabel('Binary Weights (%)'); ax.set_xlabel('Layer')
ax.set_ylim(0, 105)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
layer_chart = save_dir / 'swin_s_layer_analysis.png'
plt.savefig(layer_chart, dpi=130, bbox_inches='tight')
plt.show()
print(f"\n Layer chart saved → {layer_chart}")


PER-LAYER APB ANALYSIS

Layer Name                                                  Params    Alpha    Delta   Binary%
--------------------------------------------------------------------------------------------
  features.1.0.attn.qkv                                     27,648   0.0440   0.2073    99.07%
  features.1.0.attn.proj                                     9,216   0.0368   0.1511    99.65%
  features.1.0.mlp.0                                        36,864   0.0280   0.1419    99.04%
  features.1.0.mlp.3                                        36,864   0.0243   0.1204    99.22%
  features.1.1.attn.qkv                                     27,648   0.0528   0.2103    99.70%
  features.1.1.attn.proj                                     9,216   0.0366   0.1437    99.76%
  features.1.1.mlp.0                                        36,864   0.0332   0.1461    99.60%
  features.1.1.mlp.3                                        36,864   0.0325   0.1432    99.65%
  features.2.reduction      

C:\Users\user\AppData\Local\Temp\ipykernel_26564\443437736.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:

# ============================================================================
# CELL 15: Pruning Capability Analysis
# Swin-S uses only Linear layers in its Transformer blocks.
# Conv2d filter merging from original APB code is not applicable here.
# APB binarization acts as "soft pruning": reduces all weights to ±1,
# discarding magnitude info entirely (equivalent to 1-bit quantization).
# ============================================================================
import numpy as np

print("=" * 80)
print("PRUNING CAPABILITY ANALYSIS")
print("=" * 80)


def collect_linear_weights(model, apb=False):
    all_w = []
    for name, module in model.named_modules():
        if apb and isinstance(module, APBLayer):
            w = module.get_effective_weight().detach().cpu().numpy().flatten()
            all_w.append(w)
        elif not apb and isinstance(module, nn.Linear):
            w = module.weight.detach().cpu().numpy().flatten()
            all_w.append(w)
    return np.concatenate(all_w) if all_w else np.array([])


print("\nCollecting baseline Linear layer weights...")
baseline_w = collect_linear_weights(model_baseline, apb=False)
print(f"  Total: {len(baseline_w):,} weights across all Linear layers")

print("Collecting APB effective (binarized) weights...")
apb_eff_w = collect_linear_weights(model_apb, apb=True)
print(f"  Total: {len(apb_eff_w):,}")

# ── Magnitude pruning simulation on baseline ──────────────────────────────────
print("\nMagnitude pruning simulation (baseline Swin-S):")
print(f"  {'Threshold':>12}  {'Prunable weights':>18}  {'Pct':>8}")
print("  " + "-" * 44)
thresholds = [0.01, 0.05, 0.10, 0.20, 0.50]
pruning_rows = []
for t in thresholds:
    n = int(np.sum(np.abs(baseline_w) < t))
    pct = n / len(baseline_w) * 100
    pruning_rows.append((t, n, pct))
    print(f"  |w| < {t:.2f}      {n:>18,}  {pct:>7.2f}%")

# ── APB binary weight breakdown ───────────────────────────────────────────────
n_pos  = int(np.sum(apb_eff_w > 0))
n_neg  = int(np.sum(apb_eff_w < 0))
n_zero = int(np.sum(apb_eff_w == 0))
total  = len(apb_eff_w)

print(f"\nAPB effective weight distribution:")
print(f"  +1 :  {n_pos:>12,}  ({n_pos / total * 100:.2f}%)")
print(f"  -1 :  {n_neg:>12,}  ({n_neg / total * 100:.2f}%)")
print(f"   0 :  {n_zero:>12,}  ({n_zero / total * 100:.2f}%)")
print(f"\n  {(n_pos + n_neg) / total * 100:.1f}% of weights reduced to 1-bit sign.")
print("  Magnitude info fully discarded — equivalent to near-zero magnitude pruning.")

# ── Visualization ─────────────────────────────────────────────────────────────
fig_p, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(np.clip(baseline_w, -0.5, 0.5), bins=120,
             color='steelblue', alpha=0.75, edgecolor='none')
axes[0].set_title('Baseline Swin-S - Linear Weight Distribution')
axes[0].set_xlabel('Weight value')
axes[0].set_ylabel('Count')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.6, linewidth=1)
axes[0].grid(True, alpha=0.3)

vals_arr = np.array([-1.0, 1.0])
counts_arr = np.array([n_neg, n_pos])
axes[1].bar(vals_arr, counts_arr, width=0.3, color=['#e74c3c', '#2ecc71'],
            alpha=0.85, edgecolor='gray', linewidth=0.5)
axes[1].set_title('APB Swin-S - Binarized Weight Distribution')
axes[1].set_xlabel('Weight value')
axes[1].set_ylabel('Count')
axes[1].set_xticks([-1, 1])
axes[1].set_xticklabels(['-1', '+1'])
for v, c in zip(vals_arr, counts_arr):
    axes[1].text(v, c * 1.01, f'{c:,}\n({c / total * 100:.1f}%)',
                 ha='center', va='bottom', fontsize=8)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
pruning_chart = save_dir / 'swin_s_pruning_analysis.png'
plt.savefig(pruning_chart, dpi=130, bbox_inches='tight')
plt.show()
print(f"\nPruning analysis saved -> {pruning_chart}")

# Store for report cell
pruning_stats = {
    'thresholds': pruning_rows,
    'n_pos': n_pos, 'n_neg': n_neg, 'n_zero': n_zero, 'total': total,
}


PRUNING CAPABILITY ANALYSIS

  Total: 48,668,160 weights across all Linear layers
  Total: 48,660,480

Magnitude pruning simulation (baseline Swin-S):
     Threshold    Prunable weights       Pct
  --------------------------------------------
  |w| < 0.01               8,164,598    16.78%
  |w| < 0.05              33,835,303    69.52%
  |w| < 0.10              46,160,415    94.85%
  |w| < 0.20              48,616,751    99.89%
  |w| < 0.50              48,665,946   100.00%

APB effective weight distribution:
  +1 :    24,289,632  (49.92%)
  -1 :    24,370,848  (50.08%)
   0 :             0  (0.00%)

  100.0% of weights reduced to 1-bit sign.
  Magnitude info fully discarded — equivalent to near-zero magnitude pruning.

Pruning analysis saved -> checkpoints\swin_s_pruning_analysis.png


C:\Users\user\AppData\Local\Temp\ipykernel_26564\3914720653.py:88: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# ============================================================================
# CELL 16: COCO Speed Benchmark — All 4 Models
# ============================================================================
# Measures inference throughput on a COCO val2017 image sample.
# Classification accuracy is NOT reported: model fine-tuned on ImageNette
# (10 classes); COCO covers 80 detection categories — categories do not align.
# Speed (latency / throughput) is the valid cross-dataset metric.
# ============================================================================
import urllib.request
import time as _time
import json as _json
from pathlib import Path
from PIL import Image

print("="*80)
print("COCO SPEED BENCHMARK — Baseline / APB / FIMA-Q / APB+FIMA-Q")
print("="*80)

COCO_DIR = Path('./data/coco_sample')
COCO_DIR.mkdir(parents=True, exist_ok=True)

COCO_VAL_IDS = [
    139, 285, 632, 724, 776, 785, 802, 1000,
    1268, 1296, 1353, 1425, 1490, 1584, 1669, 2006,
    2149, 2153, 2261, 2299, 2431, 2473, 2654, 2673,
    3845, 4134, 4395, 5001, 5037, 5076,
]

print(f"\nPreparing {len(COCO_VAL_IDS)} COCO val2017 images ...")
coco_paths = []
for img_id in COCO_VAL_IDS:
    out = COCO_DIR / f"{img_id:012d}.jpg"
    if not out.exists():
        url = f"http://images.cocodataset.org/val2017/{img_id:012d}.jpg"
        try:
            urllib.request.urlretrieve(url, out)
        except Exception:
            continue
    if out.exists():
        coco_paths.append(out)

coco_tensors = []
for p in coco_paths:
    try:
        coco_tensors.append(val_transform(Image.open(p).convert('RGB')))
    except Exception:
        pass

if len(coco_tensors) < 4:
    print("  No internet access — using synthetic 224x224 tensors (COCO fallback)")
    coco_tensors = [torch.randn(3, 224, 224) for _ in range(20)]
    coco_source  = "synthetic (224x224, COCO fallback)"
else:
    coco_source = f"COCO val2017 ({len(coco_tensors)} downloaded images)"

COCO_BATCH = min(16, len(coco_tensors))
coco_batch = torch.stack(coco_tensors[:COCO_BATCH]).to(device)
print(f"  Source: {coco_source}  |  batch={COCO_BATCH}")

# ── Benchmark helper ──────────────────────────────────────────────────────
def coco_speed(model, batch, n_runs=12, warmup=2):
    model.eval()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = _time.time()
            _ = model(batch)
            times.append((_time.time() - t0) * 1000)
    t = times[warmup:]
    return float(np.mean(t)), float(np.std(t)), COCO_BATCH / (float(np.mean(t)) / 1000)

print("\nRunning COCO speed benchmark ...")
avg_base, std_base, tput_base = coco_speed(model_baseline,   coco_batch)
avg_apb,  std_apb,  tput_apb  = coco_speed(model_apb,        coco_batch)
avg_fq,   std_fq,   tput_fq   = coco_speed(model_fimaq,      coco_batch)
avg_afq,  std_afq,  tput_afq  = coco_speed(model_apb_fimaq,  coco_batch)

# ── Print results ─────────────────────────────────────────────────────────
print(f"\nCOCO Speed Results (batch={COCO_BATCH}):")
header = f"  {'':30} {'Baseline':>11} {'APB':>11} {'FIMA-Q':>11} {'APB+FIMA-Q':>12}"
print(header)
print("  " + "-"*77)
def prow(lbl, *vals): print(f"  {lbl:<30} " + "  ".join(f"{v:>11}" for v in vals))
prow("Avg inference (ms/batch)",
     f"{avg_base:.2f}", f"{avg_apb:.2f}", f"{avg_fq:.2f}", f"{avg_afq:.2f}")
prow("Std dev (ms)",
     f"{std_base:.2f}", f"{std_apb:.2f}", f"{std_fq:.2f}", f"{std_afq:.2f}")
prow("Throughput (img/s)",
     f"{tput_base:.1f}", f"{tput_apb:.1f}", f"{tput_fq:.1f}", f"{tput_afq:.1f}")
prow("Speedup vs Baseline",
     "1.00x",
     f"{avg_base/avg_apb:.2f}x",
     f"{avg_base/avg_fq:.2f}x",
     f"{avg_base/avg_afq:.2f}x")

# ── Cross-dataset speed summary ───────────────────────────────────────────
print(f"\nCross-dataset speed comparison (ImageNette val vs COCO sample):")
h2 = f"  {'':26} {'Baseline':>10} {'APB':>10} {'FIMA-Q':>10} {'APB+FQ':>10}"
print(h2); print("  " + "-"*68)
for ds_name, bs, ap, fq, afq in [
    ('ImageNette val (ms)',
     baseline_res_eval['avg_inference_time']*1e3,
     apb_res_eval['avg_inference_time']*1e3,
     fimaq_res_eval['avg_inference_time']*1e3,
     apb_fimaq_res_eval['avg_inference_time']*1e3),
    ('COCO sample (ms)', avg_base, avg_apb, avg_fq, avg_afq),
]:
    print(f"  {ds_name:<26} {bs:>10.2f} {ap:>10.2f} {fq:>10.2f} {afq:>10.2f}")

print(f"\nNote: Accuracy NOT reported for COCO.")
print(f"Model trained on ImageNette (10 classes); COCO=80 categories. Speed only.")

# ── Save ─────────────────────────────────────────────────────────────────
coco_results = {
    'source': coco_source, 'batch_size': COCO_BATCH,
    'baseline':   {'ms': round(avg_base, 2), 'std': round(std_base, 2), 'tput': round(tput_base, 1)},
    'apb':        {'ms': round(avg_apb,  2), 'std': round(std_apb,  2), 'tput': round(tput_apb,  1),
                   'speedup': round(avg_base/avg_apb, 2)},
    'fimaq':      {'ms': round(avg_fq,   2), 'std': round(std_fq,   2), 'tput': round(tput_fq,   1),
                   'speedup': round(avg_base/avg_fq,  2)},
    'apb_fimaq':  {'ms': round(avg_afq,  2), 'std': round(std_afq,  2), 'tput': round(tput_afq,  1),
                   'speedup': round(avg_base/avg_afq, 2)},
}
coco_json = save_dir / 'coco_speed_results_4model.json'
with open(coco_json, 'w') as f:
    _json.dump(coco_results, f, indent=2)
print(f"\nCOCO speed results saved -> {coco_json}")

# ── Bar chart ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
fig_c, ax_c = plt.subplots(figsize=(9, 5))
labels_c = ['Baseline', 'APB', 'FIMA-Q', 'APB+FIMA-Q']
tputs_c  = [tput_base, tput_apb, tput_fq, tput_afq]
cols_c   = [COLORS['baseline'], COLORS['apb'], COLORS['fimaq'], COLORS['apb_fimaq']]
bars_c   = ax_c.bar(labels_c, tputs_c, color=cols_c, alpha=0.88, edgecolor='black', lw=0.6)
for b in bars_c:
    ax_c.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
              f'{b.get_height():.1f}', ha='center', va='bottom', fontsize=10)
ax_c.set_title(f'COCO Throughput — 4 Models (batch={COCO_BATCH})', fontsize=13)
ax_c.set_ylabel('Images / second'); ax_c.grid(True, alpha=0.3, axis='y')
chart_c = save_dir / 'coco_throughput_4model.png'
fig_c.tight_layout()
fig_c.savefig(chart_c, dpi=140, bbox_inches='tight')
plt.show()
print(f"COCO throughput chart saved -> {chart_c}")


COCO SPEED BENCHMARK

  Available: 26 images
  Source: COCO val2017 (26 images), batch=16

COCO Speed Results (batch=16):
                                     Baseline          APB
  --------------------------------------------------------
  Avg inference (ms/batch)            2578.54      3270.23
  Std dev (ms)                          66.01       130.13
  Throughput (img/s)                      6.2          4.9
  Speedup                                            0.79x

Cross-dataset speed comparison:
  Dataset                  Baseline (ms)     APB (ms)    Speedup
  --------------------------------------------------------------
  ImageNette val                  714.31      1281.45      0.56x
  COCO val sample                2578.54      3270.23      0.79x

Note: Classification accuracy on COCO is not reported.
Model fine-tuned on ImageNette (10 classes); COCO = 80 detection categories.
Speed is the valid comparison metric here.

COCO results saved -> checkpoints\coco_speed_results.j